# Shmoo Benchmarking for DSP


In [2]:
# Equipment setup

import pyvisa
import serial
import shlex
import subprocess
import time
import matplotlib.pyplot as plt
import numpy as np

In [4]:
# Connect the SMU
rm = pyvisa.ResourceManager("@py")

smu = rm.open_resource('TCPIP::169.254.58.10::gpib0,13::INSTR')

smu_verification = smu.query("*IDN?")
print("SourceMeter:", smu_verification)

SourceMeter: Keithley Instruments Inc., Model 2602A, 1264983, 2.1.5



In [5]:
current_limit = "500e-3"
voltage_limit = "0.85"

# Initialize SMU
smu.write("smub.reset()")
smu.write("smub.source.func = smub.OUTPUT_DCVOLTS")
#$ smu.write("smub.source.autorangev = smub.AUTORANGE_ON")
smu.write(f"smub.source.limiti = {current_limit}")
smu.write(f"smub.measure.rangei = {current_limit}")
smu.write(f"smub.source.limitv = {voltage_limit}")
smu.write(f"smub.source.levelv = {voltage_limit}")
smu.write(f"smub.source.rangev = {voltage_limit}")

# Clear and reset buffers
smu.write("smub.nvbuffer1.clear()")
smu.write("smub.nvbuffer1.collecttimestamps = 1")
smu.write("smub.nvbuffer1.collectsourcevalues = 1")
smu.write("smub.nvbuffer2.clear()")
smu.write("smub.nvbuffer2.collecttimestamps = 1")
smu.write("smub.nvbuffer2.collectsourcevalues = 1")

# Capture count / delay
smu.write("smub.measure.count = 700")
smu.write("smub.measure.nplc = 0.1")


25

In [57]:
# PLL Reference Clock (MHz)
pll_ref_clock = 50

# Clock range (MHz)
start_clk, end_clk, step_clk = 600, 1100, 50

# Voltage range (V)
start_voltage, end_voltage, step_v = 0.85, 0.85, 0.0

bmark_name = 'shmoo_bench'
cmake_name = 'shmoo_bench'

In [58]:
def update_freq(freq_mhz, target_name):
    # freq_mhz = Frequency in MHz
    # target_name = Executable name
    with open("../../platform/dsp24/freq.h", "w") as f:
        ratio = freq_mhz // pll_ref_clock
        print(f'Creating header for {freq_mhz} MHz and (mult ratio of {ratio} using PLL reference of {pll_ref_clock} MHz)')
        f.write('#ifndef __FREQ_H\n#define __FREQ_H\n')
        f.write(f'#define MTIME_FREQ     {freq_mhz}000000\n')
        f.write(f'#define SYS_CLK_FREQ   {freq_mhz}000000\n')
        f.write(f'#define SHMOO_PLL_RATIO   {ratio}\n')
        f.write("#endif")


    with open("../CMakeLists.txt", "w") as f:
        f.write(f'''add_executable({target_name}
  dsp_conv_bench/src/main.c
)\n''')
        f.write(f'target_include_directories({target_name} PUBLIC dsp_conv_bench/include)\n\n')
        f.write(f'target_link_libraries({target_name} PRIVATE \n')
        f.write('  -L${CMAKE_BINARY_DIR}/glossy -Wl,--whole-archive glossy -Wl,--no-whole-archive)\n\n')

        # Didnt work: f.write(f'set_target_properties({exec_name} PROPERTIES OUTPUT_NAME "{target_name}.elf")')

In [ ]:
# Precompiles all .elf files for future benchmarking
cur_clk = start_clk
while cur_clk <= end_clk:
    target_name = f'{cmake_name}__{cur_clk}'
    update_freq(cur_clk, target_name)
    # cmake_arg_soc = shlex.split(f"/tools/C/ee290-fa24-2/.conda-env/bin/cmake -S ./ -B ./build_shmoo/ -D CMAKE_TOOLCHAIN_FILE=./riscv-gcc.cmake -DCHIP=dsp24")
    cmake_arg_shmoo = shlex.split(f"make build TARGET={target_name} CHIP=dsp24")
    #subprocess.run(cmake_arg_soc, cwd="../../")
    subprocess.run(cmake_arg_shmoo, cwd="../../")
    
    cur_clk += step_clk

In [ ]:
while cur_v <= end_voltage: # Set the voltage 
     while current_clock <= end_clock: